In [ ]:
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

import math

# import time
# import tracemalloc
# import psutil



# ==========================================================
# Detection Parameters
# ==========================================================

MIN_SUN_AREA_RATIO = 0.0002
MAX_SUN_AREA_RATIO = 0.6 #

MIN_CIRCULARITY = 0.5
MIN_SOLIDITY = 0.2

MIN_ASPECT_RATIO = 0.60
MAX_ASPECT_RATIO = 1.4

MIN_VALUE_AFTER_OTSU = 180

LOCAL_CONTRAST_LIMIT = 2 #

TRACKING_WINDOW = 1500 #

SUN_BRIGHTNESS_RATIO = 0.98 #....
RATIO = 2
# ==========================================================
# ROI Helper
# ==========================================================

def crop_roi(frame, center, window):
    h, w = frame.shape[:2]
    cx, cy = center
    half = window // 2

    x1 = max(0, cx - half)
    y1 = max(0, cy - half)

    x2 = min(w, cx + half)
    y2 = min(h, cy + half)

    roi = frame[y1:y2, x1:x2]
    return roi, x1, y1

# ==========================================================
# Sun Detection (کامل و بدون تغییر منطق)
# ==========================================================

def detect_sun_robust(frame, offset_x=0, offset_y=0):
    if frame is None or frame.size == 0:
        return None, None, None, None

    h, w = frame.shape[:2]

    if offset_x == 0 and offset_y == 0:
        sky_limit = int(h * 0.90)
        frame = frame[:sky_limit]
        h, w = frame.shape[:2]

    total_pixels = h * w

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    value = hsv[:, :, 2]
    sat = hsv[:, :, 1] #for cloud
    blur = cv2.GaussianBlur(value, (9, 9), 0)
    # blur = value
    

    # # ----------------------------
    # # OTSU & Adaptive Threshold
    # # ----------------------------
    # _, otsu = cv2.threshold(
    #     blur,
    #     0,
    #     255,
    #     cv2.THRESH_BINARY + cv2.THRESH_OTSU
    # )

    # adaptive = cv2.adaptiveThreshold(
    #     blur,
    #     255,
    #     cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    #     cv2.THRESH_BINARY,
    #     61,
    #     -4
    # )

    # bright = cv2.inRange(
    #     value,
    #     MIN_VALUE_AFTER_OTSU,
    #     255
    # )

    # # ----------------------------
    # # Final Mask
    # # ----------------------------
    # mask = cv2.bitwise_or(otsu, adaptive)
    # mask = cv2.bitwise_and(mask, bright)

    # ----------------------------
    # Custom Threshold
    # ----------------------------
    max_val = np.max(blur)
    custom_thresh_val = int(max_val * SUN_BRIGHTNESS_RATIO)

    _, mask_v = cv2.threshold(value, custom_thresh_val, 255, cv2.THRESH_BINARY)

    blur_sat = cv2.GaussianBlur(sat, (9, 9), 0)
    min_sat = np.percentile(sat , 2)
    custom_thresh_sat = int(min_sat * RATIO)
    print(custom_thresh_sat)
    _, mask_s = cv2.threshold(sat, custom_thresh_sat , 255, cv2.THRESH_BINARY_INV)

    mask_sv = cv2.bitwise_and(mask_v, mask_s)


    #...
    kernel1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))

    mask = mask_sv
    # mask_sv = mask_v
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel2)

    
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    best = None
    best_score = -1
    best_contour = None

    # -------------------------------------------------
    # Candidate Selection
    # -------------------------------------------------
    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < total_pixels * MIN_SUN_AREA_RATIO:
            continue

        if area > total_pixels * MAX_SUN_AREA_RATIO:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter <= 0:
            continue

        circularity = (4.0 * np.pi * area) / (perimeter * perimeter)
        if circularity < MIN_CIRCULARITY:
            continue

        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        if hull_area <= 0:
            continue

        solidity = area / hull_area
        if solidity < MIN_SOLIDITY:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)
        if bh == 0:
            continue

        aspect_ratio = bw / float(bh)
        if aspect_ratio < MIN_ASPECT_RATIO or aspect_ratio > MAX_ASPECT_RATIO:
            continue

        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue

        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        object_mask = np.zeros_like(mask)
        cv2.drawContours(object_mask, [cnt], -1, 255, -1)

        mean_value = cv2.mean(value, mask=object_mask)[0]

        # ---------------------------------
        # Local Contrast
        # ---------------------------------
        ring_mask = cv2.dilate(
            object_mask,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
        )
        ring_mask = cv2.subtract(ring_mask, object_mask)

        background_value = cv2.mean(value, mask=ring_mask)[0]
        local_contrast = mean_value - background_value

        if local_contrast < LOCAL_CONTRAST_LIMIT:
            continue

        mean_bgr = cv2.mean(frame, mask=object_mask)[:3]
        (center_tmp, radius) = cv2.minEnclosingCircle(cnt)
        radius = int(radius)

        score = (
            mean_value * 1.8 +
            local_contrast * 8.0 +
            circularity * 250 +
            solidity * 250 +
            np.sqrt(area)
        )

        if score > best_score:
            best_score = score
            best_contour = cnt
            best = {
                "center": (cx + offset_x, cy + offset_y),
                "radius_est": radius,
                "pixel_count": int(area),
                "mean_bgr": mean_bgr,
                "mean_value": mean_value,
                "local_contrast": local_contrast,
                "circularity": circularity,
                "solidity": solidity,
                "score": score
            }
    
    if best is None:
        # بلور سبک برای وصل کردن لبه‌های قطع شده توسط ابر
        hough_input = cv2.GaussianBlur(mask_sv, (5, 5), 0)
        
        min_r = int(np.sqrt(total_pixels * MIN_SUN_AREA_RATIO / np.pi))
        max_r = int(np.sqrt(total_pixels * MAX_SUN_AREA_RATIO / np.pi))
        
        circles = cv2.HoughCircles(
            hough_input,
            cv2.HOUGH_GRADIENT,
            dp=1.2,
            minDist=30,
            param1=80,
            param2=10,       # حساسیت پایین‌تر برای یافتن دایره‌های تکه‌تکه
            minRadius=max(5, min_r),
            maxRadius=max_r
        )

        if circles is not None:
            circles = np.uint16(np.around(circles))
            best_hough = None
            best_hough_score = -1.0

            for c in circles[0, :]:
                cx_h, cy_h, r_h = int(c[0]), int(c[1]), int(c[2])
                
                if 0 <= cx_h < w and 0 <= cy_h < h and r_h > 0:
                    # ایجاد ماسک دایره کاندید
                    temp_circle_mask = np.zeros_like(mask_sv)
                    cv2.circle(temp_circle_mask, (cx_h, cy_h), r_h, 255, -1)
                    
                    # محاسبه تعداد پیکسل‌های روشن درون دایره
                    overlap = cv2.bitwise_and(mask_sv, temp_circle_mask)
                    bright_pixels = cv2.countNonZero(overlap)
                    
                    if bright_pixels == 0:
                        continue

                    # محاسبه مساحت دایره
                    circle_area = np.pi * (r_h ** 2)
                    
                    # تراکم پیکسل‌های روشن درون دایره (پیکسل روشن بیشتر + شعاع کمتر = اسکور بالاتر)
                    density = bright_pixels / circle_area
                    
                    # اسکور ترکیبی: پاداش به تعداد پیکسل روشن و جریمه برای شعاع بزرگتر
                    hough_score = bright_pixels * density

                    if hough_score > best_hough_score:
                        best_hough_score = hough_score
                        best_hough = (cx_h, cy_h, r_h, bright_pixels)

            if best_hough is not None:
                cx_h, cy_h, r_h, bright_pixels = best_hough
                
                # ساخت ماسک نهایی برای استخراج رنگ
                final_hough_mask = np.zeros_like(mask_sv)
                cv2.circle(final_hough_mask, (cx_h, cy_h), r_h, 255, -1)
                mean_bgr = cv2.mean(frame, mask=final_hough_mask)[:3]
                mean_val = cv2.mean(value, mask=final_hough_mask)[0]

                best = {
                    "center": (cx_h + offset_x, cy_h + offset_y),
                    "radius_est": r_h,
                    "pixel_count": bright_pixels,
                    "mean_bgr": mean_bgr,
                    "mean_value": mean_val,
                    "local_contrast": 0.0,
                    "circularity": 1.0,
                    "solidity": 1.0,
                    "score": round(best_hough_score, 2),
                    "method": "hough"
                }
    return best, value, mask, best_contour , mask_sv


# ==========================================================
# Jupyter Folder Pipeline
# ==========================================================

def process_image_folder(folder_path, valid_extensions=('*.jpg', '*.jpeg', '*.png', '*.bmp')):
    #..
    # tracemalloc.start()
    # process = psutil.Process(os.getpid())
    # start_time = time.perf_counter()
    # initial_ram = process.memory_info().rss / (1024 * 1024)
    #..
    
    image_paths = []
    for ext in valid_extensions:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))

    image_paths = sorted(image_paths)

    if not image_paths:
        print(f"هیچ تصاویری در مسیر '{folder_path}' یافت نشد.")
        return

    print(f"تعداد {len(image_paths)} تصویر جهت پردازش یافت شد.\n")

    list_original = []
    list_threshold_sv = []
    list_threshold = []
    list_final = []

    last_center = None

    for idx, path in enumerate(image_paths):
        frame = cv2.imread(path)
        if frame is None:
            continue

        h, w = frame.shape[:2]
        original_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        list_original.append((os.path.basename(path), original_rgb))

        if last_center is not None:
            roi, ox, oy = crop_roi(frame, last_center, TRACKING_WINDOW)
            sun, val_img, mask_img, cnt , mask_sv_img = detect_sun_robust(roi, ox, oy)   #woow
        else:
            sun, val_img, mask_img, cnt , mask_sv_img = detect_sun_robust(frame)
            ox, oy = 0, 0

        if sun is None and last_center is not None:
            sun, val_img, mask_img, cnt , mask_sv_img = detect_sun_robust(frame)
            ox, oy = 0, 0

        list_threshold_sv.append(mask_sv_img if mask_sv_img is not None else np.zeros((h, w), dtype=np.uint8))

        # ۲. تصویر Threshold / Mask
        list_threshold.append(mask_img if mask_img is not None else np.zeros((h, w), dtype=np.uint8))

        # ۵. نتیجه نهایی
        final_img = frame.copy()
        if sun is not None:
            last_center = sun["center"]
            cx, cy = sun["center"]
            radius = sun["radius_est"]

            cv2.circle(final_img, (cx, cy), radius, (0, 255, 255), 2)
            cv2.circle(final_img, (cx, cy), 3, (0, 0, 255), -1)
            cv2.putText(final_img, f"Sun ({cx},{cy})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
            cv2.putText(final_img, f"Score : {sun['score']:.1f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (255, 255, 0), 2)
            cv2.putText(final_img, f"Contrast : {sun['local_contrast']:.1f}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (255, 255, 255), 2)
        else:
            last_center = None

        list_final.append(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))

        #...
    # end_time = time.perf_counter()
    # current_mem, peak_mem = tracemalloc.get_traced_memory()
    # tracemalloc.stop()
    # final_ram = process.memory_info().rss / (1024 * 1024)

    # total_time = end_time - start_time
    # avg_fps = len(image_paths) / total_time if total_time > 0 else 0

    # print("==================================================")
    # print("📊 گزارش بنچمارک و مصرف منابع سیستم")
    # print("==================================================")
    # print(f"⏱️ زمان کل پردازش: {total_time:.3f} ثانیه")
    # print(f"⚡ میانگین سرعت پردازش: {avg_fps:.2f} فریم بر ثانیه (FPS)")
    # print(f"🧠 رم اولیه فرایند: {initial_ram:.2f} megabytes")
    # print(f"📈 اوج تخصیص رم (Peak Memory): {peak_mem / (1024 * 1024):.2f} megabytes")
    # print(f"💾 رم نهایی اشغال‌شده: {final_ram:.2f} megabytes")
    # print("==================================================\n")

    def display_stage(stage_title, images, is_cmap_gray=False, cols_per_row=7):
        print("==================================================")
        print(f"  مرحله: {stage_title}")
        print("==================================================")
        
        n = len(images)
        if n == 0:
            print("هیچ تصویری برای نمایش وجود ندارد.")
            return
    
        # محاسبه تعداد ستون‌ها (حداکثر ۱۰) و سطرهای مورد نیاز
        cols = min(n, cols_per_row)
        rows = math.ceil(n / cols)
    
        # تنظیم ابعاد کلی شکل متناسب با تعداد سطر و ستون
        fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
        
        # تبدیل axes به لیست یک‌بعدی برای دسترسی راحت‌تر در حلقه
        if isinstance(axes, plt.Axes):
            axes = [axes]
        else:
            axes = axes.flatten()
    
        for i, img in enumerate(images):
            if is_cmap_gray:
                axes[i].imshow(img, cmap='gray')
            else:
                axes[i].imshow(img)
                
            axes[i].set_title(f"عکس {i+1}: {list_original[i][0]}", fontsize=8)
            axes[i].axis('off')
    
        # مخفی کردن محورهای خالی در سطر آخر (اگر تعداد عکس‌ها کمتر از ۱۰ در سطر آخر باشد)
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')
    
        plt.tight_layout()
        plt.show()
    
    # فراخوانی مراحل جدید:
    display_stage("۱. تصویر Threshold ", list_threshold_sv, is_cmap_gray=True)
    display_stage("۲. تصویر Threshold / ماسک نهایی", list_threshold, is_cmap_gray=True)
    display_stage("۳. نتیجه نهایی شناسایی", list_final)


# ==========================================================
# فراخوانی تابع (بدون فاصله اضافی در ابتدای خط)
# ==========================================================
process_image_folder("C:/Users/win/Desktop/SunTrackerSplit/SunTracker-Refactor/Refactor/Images")